# 01 · Camada Bronze — Extração Incremental (Estratégias Mistas)

Nem todas as tabelas da fonte oferecem as mesmas garantias. Este notebook aplica **estratégias de extração diferentes** consoante o que cada tabela disponibiliza:

- **Snapshot diferencial** — tabelas de referência/dimensão (stockitems, colors, stockgroups, customers, buyinggroups, customercategories, cities, stateprovinces, countries, people)
- **Incremental por data** — tabelas transacionais (invoices, invoicelines, orders, orderlines, customertransactions)

A tabela `bronze._load_control` regista cada execução com o watermark ou snapshot_id usado.

## 1. Imports e ligações

In [156]:
import pandas as pd
import hashlib
from datetime import datetime, date, timezone
from sqlalchemy import create_engine, text
from dotenv import load_dotenv
import os

load_dotenv()  # carrega o ficheiro .env na mesma pasta

# -- Fonte operacional (WWI PostgreSQL) --------------------------------------
SRC_HOST     = os.getenv("SRC_HOST")
SRC_PORT     = int(os.getenv("SRC_PORT", 5432))
SRC_DB       = os.getenv("SRC_DB")
SRC_USER     = os.getenv("SRC_USER")
SRC_PASSWORD = os.getenv("SRC_PASSWORD")

# -- Data Warehouse ----------------------------------------------------------
DWH_HOST     = os.getenv("DWH_HOST")
DWH_PORT     = int(os.getenv("DWH_PORT", 5432))
DWH_DB       = os.getenv("DWH_DB")
DWH_USER     = os.getenv("DWH_USER")
DWH_PASSWORD = os.getenv("DWH_PASSWORD")

engine_src = create_engine(
    f"postgresql+psycopg2://{SRC_USER}:{SRC_PASSWORD}@{SRC_HOST}:{SRC_PORT}/{SRC_DB}"
)
engine_dwh = create_engine(
    f"postgresql+psycopg2://{DWH_USER}:{DWH_PASSWORD}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}"
)

print("✓ Engines criados.")
print(f"  SRC: {SRC_USER}@{SRC_HOST}:{SRC_PORT}/{SRC_DB}")
print(f"  DWH: {DWH_USER}@{DWH_HOST}:{DWH_PORT}/{DWH_DB}")

✓ Engines criados.
  SRC: dss@postgres2.ipca.pt:5432/wwi
  DWH: postgres@localhost:5432/wwi_dw


## 2. Configuração das tabelas por estratégia

In [158]:
# Tabelas extraídas por snapshot diferencial (dimensões / referência)
SNAPSHOT_TABLES = [
    {"src_table": "stockitems",         "bronze_table": "stockitems",         "pk": ["stockitemid"]},
    {"src_table": "colors",             "bronze_table": "colors",             "pk": ["colorid"]},
    {"src_table": "stockgroups",        "bronze_table": "stockgroups",        "pk": ["stockgroupid"]},
    {"src_table": "customers",          "bronze_table": "customers",          "pk": ["customerid"]},
    {"src_table": "buyinggroups",       "bronze_table": "buyinggroups",       "pk": ["buyinggroupid"]},
    {"src_table": "customercategories", "bronze_table": "customercategories", "pk": ["customercategoryid"]},
    {"src_table": "cities",             "bronze_table": "cities",             "pk": ["cityid"]},
    {"src_table": "stateprovinces",     "bronze_table": "stateprovinces",     "pk": ["stateprovinceid"]},
    {"src_table": "countries",          "bronze_table": "countries",          "pk": ["countryid"]},
    {"src_table": "people",             "bronze_table": "people",             "pk": ["personid"]},
]

# Tabelas extraídas incrementalmente por data
# Para invoices/invoicelines/orders/orderlines usa-se invoicedate / orderdate
DATE_TABLES = [
    {
        "src_table":     "invoices",
        "bronze_table":  "invoices",
        "watermark_col": "invoicedate",
        "pk":            ["invoiceid"],
    },
    {
        "src_table":     "invoicelines",
        "bronze_table":  "invoicelines",
        "watermark_col": "invoicedate",  # herdada via JOIN com invoices
        "pk":            ["invoicelineid"],
    },
    {
        "src_table":     "orders",
        "bronze_table":  "orders",
        "watermark_col": "orderdate",
        "pk":            ["orderid"],
    },
    {
        "src_table":     "orderlines",
        "bronze_table":  "orderlines",
        "watermark_col": "orderdate",    # herdada via JOIN com orders
        "pk":            ["orderlineid"],
    },
    {
        "src_table":     "customertransactions",
        "bronze_table":  "customertransactions",
        "watermark_col": "transactiondate",
        "pk":            ["customertransactionid"],
    },
]

print("Estratégia snapshot diferencial:")
for t in SNAPSHOT_TABLES:
    print(f"  {t['src_table']:<28}  pk: {t['pk']}")

print("\nEstratégia incremental por data:")
for t in DATE_TABLES:
    print(f"  {t['src_table']:<28}  pk: {t['pk']}  watermark: {t['watermark_col']}")

Estratégia snapshot diferencial:
  stockitems                    pk: ['stockitemid']
  colors                        pk: ['colorid']
  stockgroups                   pk: ['stockgroupid']
  customers                     pk: ['customerid']
  buyinggroups                  pk: ['buyinggroupid']
  customercategories            pk: ['customercategoryid']
  cities                        pk: ['cityid']
  stateprovinces                pk: ['stateprovinceid']
  countries                     pk: ['countryid']
  people                        pk: ['personid']

Estratégia incremental por data:
  invoices                      pk: ['invoiceid']  watermark: invoicedate
  invoicelines                  pk: ['invoicelineid']  watermark: invoicedate
  orders                        pk: ['orderid']  watermark: orderdate
  orderlines                    pk: ['orderlineid']  watermark: orderdate
  customertransactions          pk: ['customertransactionid']  watermark: transactiondate


## 3. Tabela de controlo

A `bronze._load_control` serve ambas as estratégias:
- Para as tabelas de snapshot, `snapshot_id` incrementa a cada execução
- Para as tabelas por data, `snapshot_id = NULL` e `watermark_date` regista a última data carregada

In [159]:
DDL_CONTROL = """
    CREATE TABLE IF NOT EXISTS bronze._load_control (
        table_name      VARCHAR(100)  NOT NULL,
        strategy        VARCHAR(20)   NOT NULL,
        snapshot_id     INT,
        watermark_date  DATE,
        loaded_at       TIMESTAMP     NOT NULL,
        rows_total      INT,
        rows_inserted   INT,
        rows_updated    INT,
        rows_deleted    INT,
        status          VARCHAR(20)   NOT NULL,
        PRIMARY KEY (table_name, loaded_at)
    );
"""

def table_exists(conn, schema: str, table: str) -> bool:
    sql = """
        SELECT EXISTS (
            SELECT 1 FROM information_schema.tables
            WHERE table_schema = :schema AND table_name = :tname
        )
    """
    return conn.execute(text(sql), {"schema": schema, "tname": table}).scalar()


with engine_dwh.begin() as conn:
    already_existed = table_exists(conn, "bronze", "_load_control")
    conn.execute(text(DDL_CONTROL))

if already_existed:
    print("✓ Tabela bronze._load_control já existe — ignorado.")
else:
    print("✓ Tabela bronze._load_control criada.")

✓ Tabela bronze._load_control criada.


## 4. Criação das tabelas bronze

As tabelas de dimensão (snapshot) incluem `_snapshot_id` e `_change_op`.  
As tabelas transacionais (por data) não precisam desses campos — cada linha é sempre nova.

In [160]:
SNAP_META = """
    _snapshot_id  INT          NOT NULL,
    _loaded_at    TIMESTAMP    NOT NULL,
    _change_op    VARCHAR(10)  NOT NULL,
    _source       VARCHAR(100) NOT NULL
"""

DATE_META = """
    _loaded_at    TIMESTAMP    NOT NULL,
    _source       VARCHAR(100) NOT NULL
"""

DDL_BRONZE = {
# ── Dimensões (snapshot) ───────────────────────────────────────────────────────
"stockitems": f"""
    CREATE TABLE IF NOT EXISTS bronze.stockitems (
        stockitemid        INT            NOT NULL,
        stockitemname      VARCHAR(255)   NOT NULL,
        supplierid         INT,
        colorid            INT,
        unitpackageid      INT,
        outerpackageid     INT,
        brand              VARCHAR(100),
        size               VARCHAR(50),
        leadtimedays       INT,
        quantityperouter   INT,
        ischillerstock     INT,
        barcode            VARCHAR(50),
        taxrate            NUMERIC(18,3),
        unitprice          NUMERIC(18,2),
        recommendedretailprice NUMERIC(18,2),
        typicalweightperunit NUMERIC(18,3),
        marketingcomments  TEXT,
        internalcomments   TEXT,
        photo              BYTEA,
        customfields       TEXT,
        tags               TEXT,
        searchdetails      TEXT,
        {SNAP_META}
    );
""",
"colors": f"CREATE TABLE IF NOT EXISTS bronze.colors (colorid INT NOT NULL, colorname VARCHAR(50) NOT NULL, {SNAP_META});",
"stockgroups": f"CREATE TABLE IF NOT EXISTS bronze.stockgroups (stockgroupid INT NOT NULL, stockgroupname VARCHAR(100) NOT NULL, {SNAP_META});",
"customers": f"""
    CREATE TABLE IF NOT EXISTS bronze.customers (
        customerid             INT            NOT NULL,
        customername           VARCHAR(255)   NOT NULL,
        billtocustomerid       INT,
        customercategoryid     INT,
        buyinggroupid          INT,
        primarycontactpersonid INT,
        alternatecontactpersonid INT,
        deliverymethodid       INT,
        deliverycityid         INT,
        postalcityid           INT,
        creditlimit            NUMERIC(18,2),
        accountopeneddate      DATE,
        standarddiscountpercentage NUMERIC(18,3),
        isstatementsent        INT,
        isoncredithold         INT,
        paymentdays            INT,
        phonenumber            VARCHAR(50),
        faxnumber              VARCHAR(50),
        deliveryrun            VARCHAR(50),
        runposition            VARCHAR(50),
        websiteurl             VARCHAR(255),
        deliveryaddressline1   VARCHAR(255),
        deliveryaddressline2   VARCHAR(255),
        deliverypostalcode     VARCHAR(20),
        deliverylocation       TEXT,
        postaladdressline1     VARCHAR(255),
        postaladdressline2     VARCHAR(255),
        postalpostalcode       VARCHAR(20),
        {SNAP_META}
    );
""",
"buyinggroups": f"CREATE TABLE IF NOT EXISTS bronze.buyinggroups (buyinggroupid INT NOT NULL, buyinggroupname VARCHAR(100) NOT NULL, {SNAP_META});",
"customercategories": f"CREATE TABLE IF NOT EXISTS bronze.customercategories (customercategoryid INT NOT NULL, customercategoryname VARCHAR(100) NOT NULL, {SNAP_META});",
"cities": f"CREATE TABLE IF NOT EXISTS bronze.cities (cityid INT NOT NULL, cityname VARCHAR(100) NOT NULL, stateprovinceid INT NOT NULL, latestrecordedpopulation BIGINT, location TEXT, {SNAP_META});",
"stateprovinces": f"CREATE TABLE IF NOT EXISTS bronze.stateprovinces (stateprovinceid INT NOT NULL, stateprovincecode VARCHAR(10), stateprovincename VARCHAR(100) NOT NULL, countryid INT NOT NULL, salesterritory VARCHAR(100), latestrecordedpopulation BIGINT, {SNAP_META});",
"countries": f"CREATE TABLE IF NOT EXISTS bronze.countries (countryid INT NOT NULL, ountryname VARCHAR(100) NOT NULL, formalname VARCHAR(100), isocountrycode VARCHAR(10), isoalpha3code VARCHAR(10), continent VARCHAR(50), region VARCHAR(50), subregion VARCHAR(50), latestrecordedpopulation BIGINT, {SNAP_META});",
"people": f"""
    CREATE TABLE IF NOT EXISTS bronze.people (
        personid           INT           NOT NULL,
        fullname           VARCHAR(255)  NOT NULL,
        preferredname      VARCHAR(255),
        searchname         VARCHAR(255),
        ispermittedtologon INT,
        logonname          VARCHAR(255),
        isexternallogonprovider INT,
        hashedpassword     TEXT,
        issystemuser       INT,
        isemployee         INT,
        issalesperson      INT,
        userpreferences    TEXT,
        phonenumber        VARCHAR(50),
        faxnumber          VARCHAR(50),
        emailaddress       VARCHAR(255),
        photo              BYTEA,
        customfields       TEXT,
        otherlanguages     TEXT,
        {SNAP_META}
    );
""",

# ── Transacionais (incremental por data) ──────────────────────────────────────
"invoices": f"""
    CREATE TABLE IF NOT EXISTS bronze.invoices (
        invoiceid              INT            NOT NULL,
        customerid             INT            NOT NULL,
        billtocustomerid       INT,
        orderid                INT,
        deliverymethodid       INT,
        contactpersonid        INT,
        accountspersonid       INT,
        salespersonpersonid    INT,
        packedbypersonid       INT,
        invoicedate            DATE           NOT NULL,
        deliveryrun            VARCHAR(50),
        runposition            VARCHAR(50),
        returneddeliverydata   TEXT,
        confirmeddeliverytime  TIMESTAMP,
        confirmedreceivedby    VARCHAR(255),
        {DATE_META}
    );
""",
"invoicelines": f"""
    CREATE TABLE IF NOT EXISTS bronze.invoicelines (
        invoicelineid    INT            NOT NULL,
        invoiceid        INT            NOT NULL,
        stockitemid      INT            NOT NULL,
        description      VARCHAR(255),
        packagetypeid    INT,
        quantity         INT            NOT NULL,
        unitprice        NUMERIC(18,2),
        taxrate          NUMERIC(18,3),
        taxamount        NUMERIC(18,2),
        lineprofit       NUMERIC(18,2),
        extendedprice    NUMERIC(18,2),
        invoicedate      DATE           NOT NULL,
        {DATE_META}
    );
""",
"orders": f"""
    CREATE TABLE IF NOT EXISTS bronze.orders (
        orderid                INT            NOT NULL,
        customerid             INT            NOT NULL,
        salespersonpersonid    INT,
        pickedbypersonid       INT,
        contactpersonid        INT,
        backorderorderid       INT,
        orderdate              DATE           NOT NULL,
        expecteddeliverydate   DATE,
        isundersupplybackordered INT,
        comments               TEXT,
        deliveryinstructions   TEXT,
        internalcomments       TEXT,
        pickingcompletedwhen   TIMESTAMP,
        {DATE_META}
    );
""",
"orderlines": f"""
    CREATE TABLE IF NOT EXISTS bronze.orderlines (
        orderlineid       INT            NOT NULL,
        orderid           INT            NOT NULL,
        stockitemid       INT            NOT NULL,
        description       VARCHAR(255),
        packagetypeid     INT,
        quantity          INT            NOT NULL,
        unitprice         NUMERIC(18,2),
        taxrate           NUMERIC(18,3),
        pickedquantity    INT,
        orderdate         DATE           NOT NULL,
        {DATE_META}
    );
""",
"customertransactions": f"""
    CREATE TABLE IF NOT EXISTS bronze.customertransactions (
        customertransactionid   INT            NOT NULL,
        customerid              INT            NOT NULL,
        transactiontypeid       INT            NOT NULL,
        invoiceid               INT,
        paymentmethodid         INT,
        transactiondate         DATE           NOT NULL,
        amountexcludingtax      NUMERIC(18,2),
        taxamount               NUMERIC(18,2),
        transactionamount       NUMERIC(18,2)  NOT NULL,
        outstandingbalance      NUMERIC(18,2)  NOT NULL,
        finalizationdate        DATE,
        isfinalized             INT,
        {DATE_META}
    );
"""
}

with engine_dwh.begin() as conn:
    created, skipped = [], []
    for name, ddl in DDL_BRONZE.items():
        existed = table_exists(conn, "bronze", name)
        conn.execute(text(ddl))
        (skipped if existed else created).append(name)

## 5. Funções — estratégia snapshot diferencial

Aplicam-se apenas às tabelas de dimensão. Comparam o snapshot actual com o anterior e guardam apenas as linhas com alterações (`INSERT`, `UPDATE`, `DELETE`).

In [161]:
def get_next_snapshot_id(table_name: str) -> int:
    sql = """
        SELECT COALESCE(MAX(snapshot_id), 0) + 1
        FROM   bronze._load_control
        WHERE  table_name = :tname
    """
    with engine_dwh.connect() as conn:
        return conn.execute(text(sql), {"tname": table_name}).scalar()


def get_last_snapshot(table_name: str, pk: list) -> pd.DataFrame:
    """Lê o snapshot mais recente do bronze (apenas registos ativos)."""
    sql = f"""
        SELECT *
        FROM   bronze.{table_name}
        WHERE  _snapshot_id = (SELECT MAX(_snapshot_id) FROM bronze.{table_name})
          AND  _change_op != 'DELETE'
    """
    try:
        with engine_dwh.connect() as conn:
            df = pd.read_sql(text(sql), conn)
        meta_cols = ["_snapshot_id", "_loaded_at", "_change_op", "_source"]
        return df.drop(columns=[c for c in meta_cols if c in df.columns])
    except Exception:
        return pd.DataFrame()


def compute_row_hash(df: pd.DataFrame, cols: list) -> pd.Series:
    return df[cols].astype(str).apply(
        lambda row: hashlib.md5("|".join(row).encode()).hexdigest(), axis=1
    )


def diff_snapshots(df_new: pd.DataFrame,
                   df_prev: pd.DataFrame,
                   pk: list) -> pd.DataFrame:
    """Compara dois snapshots e devolve apenas as linhas alteradas com _change_op."""
    df_new  = df_new.copy().set_index(pk)

    if df_prev.empty:
        result = df_new.reset_index()
        result["_change_op"] = "INSERT"
        return result

    df_prev = df_prev.copy().set_index(pk)
    common_cols = [c for c in df_new.columns if c in df_prev.columns]

    hash_new  = compute_row_hash(df_new.reset_index(),  pk + common_cols)
    hash_prev = compute_row_hash(df_prev.reset_index(), pk + common_cols)
    hash_new.index  = df_new.index
    hash_prev.index = df_prev.index

    changed_rows = []

    new_keys = df_new.index.difference(df_prev.index)
    if len(new_keys):
        ins = df_new.loc[new_keys].reset_index()
        ins["_change_op"] = "INSERT"
        changed_rows.append(ins)

    common_keys  = df_new.index.intersection(df_prev.index)
    updated_keys = common_keys[hash_new.loc[common_keys] != hash_prev.loc[common_keys]]
    if len(updated_keys):
        upd = df_new.loc[updated_keys].reset_index()
        upd["_change_op"] = "UPDATE"
        changed_rows.append(upd)

    deleted_keys = df_prev.index.difference(df_new.index)
    if len(deleted_keys):
        dlt = df_prev.loc[deleted_keys].reset_index().reindex(
            columns=df_new.reset_index().columns
        )
        dlt["_change_op"] = "DELETE"
        changed_rows.append(dlt)

    return pd.concat(changed_rows, ignore_index=True) if changed_rows else pd.DataFrame()


print("✓ Funções de snapshot definidas.")

✓ Funções de snapshot definidas.


## 6. Funções — estratégia incremental por data

**`get_watermark_date(table, col)`** — lê a data máxima já carregada no bronze. Na primeira execução devolve `1900-01-01`.

As tabelas `invoicelines` e `orderlines` herdam a data das respetivas tabelas-pai (`invoices`/`orders`) via JOIN, garantindo que todas as linhas de uma fatura/encomenda nova são carregadas em conjunto.

In [162]:
DATE_ZERO = date(1900, 1, 1)

def get_watermark_date(bronze_table: str, date_col: str) -> date:
    sql = f"SELECT MAX({date_col}) FROM bronze.{bronze_table}"
    try:
        with engine_dwh.connect() as conn:
            result = conn.execute(text(sql)).scalar()
        return result if result is not None else DATE_ZERO
    except Exception:
        return DATE_ZERO

def extract_invoices(watermark: date) -> pd.DataFrame:
    sql = """
        SELECT invoiceid, customerid, billtocustomerid, orderid, deliverymethodid,
               contactpersonid, accountspersonid, salespersonpersonid, packedbypersonid,
               invoicedate, deliveryrun, runposition,
               returneddeliverydata, confirmeddeliverytime, confirmedreceivedby
        FROM   public.invoices
        WHERE  invoicedate > :wm
        ORDER  BY invoicedate
    """
    with engine_src.connect() as conn:
        return pd.read_sql(text(sql), conn, params={"wm": watermark})

def extract_invoicelines(new_invoice_ids: list) -> pd.DataFrame:
    if not new_invoice_ids:
        return pd.DataFrame()
    sql = """
        SELECT il.invoicelineid, il.invoiceid, il.stockitemid, il.description,
               il.packagetypeid, il.quantity, il.unitprice, il.taxrate,
               il.taxamount, il.lineprofit, il.extendedprice,
               i.invoicedate
        FROM   public.invoicelines il
        JOIN   public.invoices     i  ON i.invoiceid = il.invoiceid
        WHERE  il.invoiceid = ANY(:ids)
    """
    with engine_src.connect() as conn:
        return pd.read_sql(text(sql), conn, params={"ids": new_invoice_ids})

def extract_orders(watermark: date) -> pd.DataFrame:
    sql = """
        SELECT orderid, customerid, salespersonpersonid, pickedbypersonid,
               contactpersonid, backorderorderid, orderdate, expecteddeliverydate,
               isundersupplybackordered, comments,
               deliveryinstructions, internalcomments, pickingcompletedwhen
        FROM   public.orders
        WHERE  orderdate > :wm
        ORDER  BY orderdate
    """
    with engine_src.connect() as conn:
        return pd.read_sql(text(sql), conn, params={"wm": watermark})

def extract_orderlines(new_order_ids: list) -> pd.DataFrame:
    if not new_order_ids:
        return pd.DataFrame()
    sql = """
        SELECT ol.orderlineid, ol.orderid, ol.stockitemid, ol.description,
               ol.packagetypeid, ol.quantity, ol.unitprice, ol.taxrate,
               ol.pickedquantity,
               o.orderdate
        FROM   public.orderlines ol
        JOIN   public.orders     o   ON o.orderid = ol.orderid
        WHERE  ol.orderid = ANY(:ids)
    """
    with engine_src.connect() as conn:
        return pd.read_sql(text(sql), conn, params={"ids": new_order_ids})

def extract_customertransactions(watermark: date) -> pd.DataFrame:
    sql = """
        SELECT customertransactionid, customerid, transactiontypeid, invoiceid,
               paymentmethodid, transactiondate, amountexcludingtax, taxamount,
               transactionamount, outstandingbalance, finalizationdate, isfinalized
        FROM   public.customertransactions
        WHERE  transactiondate > :wm
        ORDER  BY transactiondate
    """
    with engine_src.connect() as conn:
        return pd.read_sql(text(sql), conn, params={"wm": watermark})

print("✓ Funções de extração por data definidas.")

✓ Funções de extração por data definidas.


## 7. Função de registo no controlo

In [163]:
def register_load(table_name: str, strategy: str, loaded_at: datetime,
                  snapshot_id=None, watermark_date=None,
                  rows_total=0, rows_inserted=0, rows_updated=0,
                  rows_deleted=0, status="SUCCESS") -> None:
    sql = """
        INSERT INTO bronze._load_control
            (table_name, strategy, snapshot_id, watermark_date, loaded_at,
             rows_total, rows_inserted, rows_updated, rows_deleted, status)
        VALUES
            (:tname, :strategy, :sid, :wm_date, :ts,
             :total, :inserted, :updated, :deleted, :status)
    """
    with engine_dwh.begin() as conn:
        conn.execute(text(sql), {
            "tname":    table_name,   "strategy": strategy,
            "sid":      snapshot_id,  "wm_date":  watermark_date,
            "ts":       loaded_at,    "total":    rows_total,
            "inserted": rows_inserted, "updated": rows_updated,
            "deleted":  rows_deleted,  "status":  status,
        })

print("✓ Função de registo definida.")

✓ Função de registo definida.


## 8. Pipeline — dimensões (snapshot diferencial)

In [164]:
run_at = datetime.now(timezone.utc).replace(tzinfo=None)

print(f"Início: {run_at.strftime('%Y-%m-%d %H:%M:%S')} UTC")
print(f"\n── Snapshot diferencial ──────────────────────────────────────")
print(f"{'Tabela':<28} {'Snap':>5} {'Total':>7} {'INSERT':>7} {'UPDATE':>7} {'DELETE':>7}")
print("-" * 65)

# Esquemas de cada tabela na fonte WWI - CORRIGIDO PARA 'public'
SRC_SCHEMAS = {
    "stockitems":          "public",
    "colors":              "public",
    "stockgroups":         "public",
    "stockitemstockgroups":"public",
    "customers":           "public",
    "buyinggroups":        "public",
    "customercategories":  "public",
    "cities":              "public",
    "stateprovinces":      "public",
    "countries":           "public",
    "people":              "public",
}

for t in SNAPSHOT_TABLES:
    src  = t["src_table"]
    dest = t["bronze_table"]
    pk   = t["pk"]
    schema = SRC_SCHEMAS.get(src, "public")
    snap_id = get_next_snapshot_id(dest)

    try:
        with engine_src.connect() as conn:
            df_new = pd.read_sql(text(f"SELECT * FROM {schema}.{src}"), conn)

        n_total = len(df_new)
        df_prev = get_last_snapshot(dest, pk)
        df_diff = diff_snapshots(df_new, df_prev, pk)

        if df_diff.empty:
            register_load(dest, "snapshot", run_at, snapshot_id=snap_id,
                          rows_total=n_total)
            print(f"{src:<28} {snap_id:>5} {n_total:>7} {'—':>7} {'—':>7} {'—':>7}  sem alterações")
            continue

        df_diff["_snapshot_id"] = snap_id
        df_diff["_loaded_at"]   = run_at
        df_diff["_source"]      = SRC_DB

        with engine_dwh.begin() as conn:
            df_diff.to_sql(dest, conn, schema="bronze", if_exists="append", index=False)

        n_i = int((df_diff["_change_op"] == "INSERT").sum())
        n_u = int((df_diff["_change_op"] == "UPDATE").sum())
        n_d = int((df_diff["_change_op"] == "DELETE").sum())

        register_load(dest, "snapshot", run_at, snapshot_id=snap_id,
                      rows_total=n_total, rows_inserted=n_i,
                      rows_updated=n_u, rows_deleted=n_d)

        print(f"{src:<28} {snap_id:>5} {n_total:>7} {n_i:>7} {n_u:>7} {n_d:>7}")

    except Exception as e:
        register_load(dest, "snapshot", run_at, snapshot_id=snap_id, status="ERROR")
        print(f"{src:<28} ERRO: {str(e)[:60]}")

print("\nSnapshot diferencial concluído.")

Início: 2026-04-16 18:07:23 UTC

── Snapshot diferencial ──────────────────────────────────────
Tabela                        Snap   Total  INSERT  UPDATE  DELETE
-----------------------------------------------------------------
stockitems                       1     227     227       0       0
colors                           1      36      36       0       0
stockgroups                      1      10      10       0       0
customers                        1     663     663       0       0
buyinggroups                     1       2       2       0       0
customercategories               1       8       8       0       0
cities                           1   37940   37940       0       0
stateprovinces                   1      53      53       0       0
countries                        1     190     190       0       0
people                           1    1111    1111       0       0

Snapshot diferencial concluído.


## 9. Pipeline — tabelas transacionais (incremental por data)

O watermark é lido diretamente da tabela bronze, não do controlo. As `invoicelines` e `orderlines` herdam os IDs das respetivas tabelas-pai já carregadas nesta execução.

In [165]:
print(f"── Incremental por data ──────────────────────────────────────")
print(f"{'Tabela':<25} {'Watermark anterior':<22} {'Novas linhas':>12}")
print("-" * 62)

run_at = datetime.now(timezone.utc).replace(tzinfo=None)

# ── invoices ──────────────────────────────────────────────────────────────────
n = 0
try:
    wm = get_watermark_date("invoices", "invoicedate")
    df_invoices = extract_invoices(wm)
    n = len(df_invoices)

    if n > 0:
        df_invoices["_loaded_at"] = run_at
        df_invoices["_source"]    = SRC_DB
        with engine_dwh.begin() as conn:
            df_invoices.to_sql("invoices", conn, schema="bronze", if_exists="append", index=False)

    new_wm = df_invoices["invoicedate"].max() if n > 0 else wm
    register_load("invoices", "incremental_date", run_at,
                  watermark_date=new_wm, rows_total=n, rows_inserted=n)
    print(f"{'invoices':<25} {str(wm):<22} {n:>12}")

except Exception as e:
    register_load("invoices", "incremental_date", run_at, status="ERROR")
    print(f"{'invoices':<25} ERRO: {str(e)[:60]}")
    df_invoices = pd.DataFrame()

# ── invoicelines ──────────────────────────────────────────────────────────────
n_il = 0
try:
    new_ids = df_invoices["invoiceid"].tolist() if not df_invoices.empty else []
    df_il   = extract_invoicelines(new_ids)
    n_il    = len(df_il)

    if n_il > 0:
        df_il["_loaded_at"] = run_at
        df_il["_source"]    = SRC_DB
        with engine_dwh.begin() as conn:
            df_il.to_sql("invoicelines", conn, schema="bronze", if_exists="append", index=False)

    register_load("invoicelines", "incremental_date", run_at,
                  watermark_date=new_wm if n > 0 else get_watermark_date("invoicelines", "invoicedate"),
                  rows_total=n_il, rows_inserted=n_il)
    print(f"{'invoicelines':<25} {str(wm):<22} {n_il:>12}")

except Exception as e:
    register_load("invoicelines", "incremental_date", run_at, status="ERROR")
    print(f"{'invoicelines':<25} ERRO: {str(e)[:60]}")

# ── orders ────────────────────────────────────────────────────────────────────
n_o = 0
try:
    wm_o = get_watermark_date("orders", "orderdate")
    df_orders = extract_orders(wm_o)
    n_o = len(df_orders)

    if n_o > 0:
        df_orders["_loaded_at"] = run_at
        df_orders["_source"]    = SRC_DB
        with engine_dwh.begin() as conn:
            df_orders.to_sql("orders", conn, schema="bronze", if_exists="append", index=False)

    new_wm_o = df_orders["orderdate"].max() if n_o > 0 else wm_o
    register_load("orders", "incremental_date", run_at,
                  watermark_date=new_wm_o, rows_total=n_o, rows_inserted=n_o)
    print(f"{'orders':<25} {str(wm_o):<22} {n_o:>12}")

except Exception as e:
    register_load("orders", "incremental_date", run_at, status="ERROR")
    print(f"{'orders':<25} ERRO: {str(e)[:60]}")
    df_orders = pd.DataFrame()

# ── orderlines ────────────────────────────────────────────────────────────────
n_ol = 0
try:
    new_ids_o = df_orders["orderid"].tolist() if not df_orders.empty else []
    df_ol     = extract_orderlines(new_ids_o)
    n_ol      = len(df_ol)

    if n_ol > 0:
        df_ol["_loaded_at"] = run_at
        df_ol["_source"]    = SRC_DB
        with engine_dwh.begin() as conn:
            df_ol.to_sql("orderlines", conn, schema="bronze", if_exists="append", index=False)

    register_load("orderlines", "incremental_date", run_at,
                  watermark_date=new_wm_o if n_o > 0 else get_watermark_date("orderlines", "orderdate"),
                  rows_total=n_ol, rows_inserted=n_ol)
    print(f"{'orderlines':<25} {str(wm_o):<22} {n_ol:>12}")

except Exception as e:
    register_load("orderlines", "incremental_date", run_at, status="ERROR")
    print(f"{'orderlines':<25} ERRO: {str(e)[:60]}")

# ── customertransactions ──────────────────────────────────────────────────────
n_ct = 0
try:
    wm_ct = get_watermark_date("customertransactions", "transactiondate")
    df_ct = extract_customertransactions(wm_ct)
    n_ct  = len(df_ct)

    if n_ct > 0:
        df_ct["_loaded_at"] = run_at
        df_ct["_source"]    = SRC_DB
        with engine_dwh.begin() as conn:
            df_ct.to_sql("customertransactions", conn, schema="bronze", if_exists="append", index=False)

    new_wm_ct = df_ct["transactiondate"].max() if n_ct > 0 else wm_ct
    register_load("customertransactions", "incremental_date", run_at,
                  watermark_date=new_wm_ct, rows_total=n_ct, rows_inserted=n_ct)
    print(f"{'customertransactions':<25} {str(wm_ct):<22} {n_ct:>12}")

except Exception as e:
    register_load("customertransactions", "incremental_date", run_at, status="ERROR")
    print(f"{'customertransactions':<25} ERRO: {str(e)[:60]}")

print("\nExtração incremental por data concluída.")

── Incremental por data ──────────────────────────────────────
Tabela                    Watermark anterior     Novas linhas
--------------------------------------------------------------
invoices                  1900-01-01                    70510
invoicelines              1900-01-01                   228265
orders                    1900-01-01                    73595
orderlines                1900-01-01                   231412
customertransactions      1900-01-01                    97147

Extração incremental por data concluída.


## 10. Consulta ao controlo de cargas

In [166]:
with engine_dwh.connect() as conn:
    df_ctrl = pd.read_sql(
        text("""
            SELECT table_name, strategy, snapshot_id, watermark_date,
                   loaded_at, rows_total, rows_inserted, rows_updated,
                   rows_deleted, status
            FROM   bronze._load_control
            ORDER  BY loaded_at DESC, table_name
        """),
        conn
    )

df_ctrl

,table_name,strategy,snapshot_id,watermark_date,loaded_at,rows_total,rows_inserted,rows_updated,rows_deleted,status
0,customertransactions,incremental_date,NaN,2020-05-31,2026-04-16 18:07:40.452424,97147,97147,0,0,SUCCESS
1,invoicelines,incremental_date,NaN,2020-05-31,2026-04-16 18:07:40.452424,228265,228265,0,0,SUCCESS
2,invoices,incremental_date,NaN,2020-05-31,2026-04-16 18:07:40.452424,70510,70510,0,0,SUCCESS
3,orderlines,incremental_date,NaN,2020-05-31,2026-04-16 18:07:40.452424,231412,231412,0,0,SUCCESS
4,orders,incremental_date,NaN,2020-05-31,2026-04-16 18:07:40.452424,73595,73595,0,0,SUCCESS
5,buyinggroups,snapshot,1.0,None,2026-04-16 18:07:23.690208,2,2,0,0,SUCCESS
6,cities,snapshot,1.0,None,2026-04-16 18:07:23.690208,37940,37940,0,0,SUCCESS
7,colors,snapshot,1.0,None,2026-04-16 18:07:23.690208,36,36,0,0,SUCCESS
8,countries,snapshot,1.0,None,2026-04-16 18:07:23.690208,190,190,0,0,SUCCESS
9,customercategories,snapshot,1.0,None,2026-04-16 18:07:23.690208,8,8,0,0,SUCCESS


## 11. Validação — contagens por tabela bronze

In [167]:
print(f"{'Tabela':<28} {'Estratégia':<22} {'Linhas no bronze':>16}")
print("-" * 70)

for t in SNAPSHOT_TABLES:
    dest = t["bronze_table"]
    with engine_dwh.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM bronze.{dest}")).scalar()
    print(f"  {dest:<26} {'snapshot diferencial':<22} {n:>16}")

for t in DATE_TABLES:
    dest = t["bronze_table"]
    with engine_dwh.connect() as conn:
        n = conn.execute(text(f"SELECT COUNT(*) FROM bronze.{dest}")).scalar()
    print(f"  {dest:<26} {'incremental por data':<22} {n:>16}")

Tabela                       Estratégia             Linhas no bronze
----------------------------------------------------------------------
  stockitems                 snapshot diferencial                227
  colors                     snapshot diferencial                 36
  stockgroups                snapshot diferencial                 10
  customers                  snapshot diferencial                663
  buyinggroups               snapshot diferencial                  2
  customercategories         snapshot diferencial                  8
  cities                     snapshot diferencial              37940
  stateprovinces             snapshot diferencial                 53
  countries                  snapshot diferencial                190
  people                     snapshot diferencial               1111
  invoices                   incremental por data              70510
  invoicelines               incremental por data             228265
  orders                     inc

## 12. Resumo

| | Dimensões | Transacionais |
|---|---|---|
| **Estratégia** | Snapshot diferencial | Incremental por data |
| **Watermark** | `_snapshot_id` máximo | `MAX(invoicedate / orderdate / transactiondate)` |
| **Tabelas** | stockitems, colors, stockgroups, customers, buyinggroups, customercategories, cities, stateprovinces, countries, people | invoices, invoicelines, orders, orderlines, customertransactions |